In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import scipy.stats as stats
import nibabel as nib
from nilearn import plotting
from rsatoolbox.inference import eval_fixed
from rsatoolbox.model import ModelFixed
from rsatoolbox.util.searchlight import get_volume_searchlight, get_searchlight_RDMs, evaluate_models_searchlight
from statsmodels.stats.multitest import fdrcorrection
from nilearn.image import new_img_like
import nilearn.image as nlimg
import seaborn as sns
from sklearn.metrics import pairwise_distances
from scipy.stats import pearsonr
from nilearn.image import resample_img

In [2]:
def upper_tri(RDM):
    """upper_
     returns the upper triangular index of an RDM

    Args:
        RDM 2Darray: squareform RDM

    Returns:
        1D array: upper triangular vector of the RDM
    """
    # returns the upper triangle
    m = RDM.shape[0]
    r, c = np.triu_indices(m, 1)
    return RDM[r, c]

# Smoothing function: Moving average
def smooth(data, window_size=50):
    return np.convolve(data, np.ones(window_size) / window_size, mode='same')


def find_significant_clusters(data, baseline_mean, baseline_std, min_consecutive=4):
    # Calculate threshold as 5 standard deviations above baseline mean
    threshold = baseline_mean + 2 * baseline_std
    
    # Find points exceeding threshold AND positive correlation

    significant_points = (data > threshold) & (data > 0)
    
    # Find clusters of consecutive significant points
    significant_clusters = []
    count = 0
    start_idx = None
    
    for i in range(len(significant_points)):
        if significant_points[i]:
            if start_idx is None:
                start_idx = i
            count += 1
        else:
            if count >= min_consecutive:
                significant_clusters.extend(range(start_idx, start_idx + count))
            count = 0
            start_idx = None
            
    # Check for significant cluster at the end
    if count >= min_consecutive:
        significant_clusters.extend(range(start_idx, start_idx + count))
        
    return np.array(significant_clusters, dtype=int)

# # Function to extract masked data and compute RDM
# def compute_rdm(brain_data, mask, mask_name):
#     """
#     Compute the RDM for masked voxel data across stimuli.

#     Parameters:
#         brain_data (numpy array): Shape (20, 60, 53, 63, 46), brain data per subject.
#         mask (numpy array): Binary mask of shape (53, 63, 46).
#         mask_name (str): Name of the mask (for logging).

#     Returns:
#         rdms (numpy array): RDM of shape (20, 60, 60) for all subjects.
#     """
#     n_subjects, n_stimuli = brain_data.shape[:2]
#     rdms = np.zeros((n_subjects, n_stimuli, n_stimuli))

#     # Flatten the mask to 1D
#     mask_flat = mask.ravel()  # Shape: (153594,)
#     total_voxels_in_mask = np.sum(mask_flat)  # Total voxels selected by mask

#     # Reshape brain data to (20, 60, num_voxels)
#     brain_data_flat = brain_data.reshape(n_subjects, n_stimuli, -1)  # Shape: (20, 60, 153594)

#     print(f"{mask_name}: Total voxel count in mask: {total_voxels_in_mask}")

#     # Apply the mask to extract relevant voxels
#     for subj in range(n_subjects):
#         masked_data = brain_data_flat[subj][:, mask_flat > 0]  # Shape: (60, num_voxels_in_mask)

#         # Check for NaN values in the masked data
#         nan_count = np.isnan(masked_data).sum()
#         if nan_count > 0:
#             print(f"Subject {subj + 1}: Masked data contains {nan_count} NaN values. "
#                   f"Total voxel count in the mask is {total_voxels_in_mask}.")

#             # Optionally replace NaNs with 0 to proceed
#             masked_data = np.nan_to_num(masked_data)

#         # Ensure there are voxels to compute RDM
#         if masked_data.shape[1] == 0:
#             raise ValueError("Mask results in zero voxels being selected.")

#         # Compute pairwise distances (RDM)
#         rdm = pairwise_distances(masked_data, metric='correlation')  # Shape: (60, 60)
#         rdms[subj] = rdm

#     return rdms

def compute_rdm(brain_data, mask, mask_name):
    """
    Compute the RDM for masked voxel data across stimuli.

    Parameters:
        brain_data (numpy array): Shape (20, 60, 53, 63, 46), brain data per subject.
        mask (numpy array): Binary mask of shape (53, 63, 46).
        mask_name (str): Name of the mask (for logging).

    Returns:
        rdms (numpy array): RDM of shape (20, 60, 60) for all subjects.
    """
    from sklearn.metrics import pairwise_distances  # Ensure this is imported

    n_subjects, n_stimuli = brain_data.shape[:2]
    rdms = np.zeros((n_subjects, n_stimuli, n_stimuli))

    # Flatten the mask to 1D
    mask_flat = mask.ravel()  # Shape: (153594,)
    total_voxels_in_mask = np.sum(mask_flat)  # Total voxels selected by mask

    # Reshape brain data to (20, 60, num_voxels)
    brain_data_flat = brain_data.reshape(n_subjects, n_stimuli, -1)  # Shape: (20, 60, 153594)

    print(f"{mask_name}: Total voxel count in mask: {total_voxels_in_mask}")

    # Loop over subjects to compute their RDMs
    for subj in range(n_subjects):
        # Extract only the voxels within the mask
        masked_data = brain_data_flat[subj][:, mask_flat > 0]  # Shape: (60, num_voxels_in_mask)

        # Check for NaN values in the masked data
        nan_count = np.isnan(masked_data).sum()
        if nan_count > 0:
            print(f"Subject {subj + 1}: Masked data contains {nan_count} NaN values. Total voxel count in mask: {total_voxels_in_mask}.")
            # Count valid voxels (columns where every stimulus value is not NaN)
            valid_voxels = np.sum(np.all(~np.isnan(masked_data), axis=0))
            if valid_voxels == 0:
                print(f"Subject {subj + 1}: All {masked_data.shape[1]} masked voxels are NaN.")
            else:
                print(f"Subject {subj + 1}: {valid_voxels} out of {masked_data.shape[1]} masked voxels are completely valid (non-NaN).")
            # Optionally, replace NaNs with 0 to proceed
            masked_data = np.nan_to_num(masked_data)
        else:
            valid_voxels = masked_data.shape[1]
            print(f"Subject {subj + 1}: All {valid_voxels} masked voxels are valid (non-NaN).")

        # Ensure there are voxels to compute the RDM
        if masked_data.shape[1] == 0:
            raise ValueError(f"Subject {subj + 1}: Mask results in zero voxels being selected.")

        # Compute the pairwise distances (RDM) using correlation as the metric
        rdm = pairwise_distances(masked_data, metric='correlation')  # Shape: (60, 60)
        rdms[subj] = rdm

    return rdms


# Function to compute EEG RDMs
def compute_eeg_rdms(eeg_arr, metric='correlation'):
    """
    Compute EEG RDMs based on the 31 EEG channels across stimuli and time points.

    Parameters:
        eeg_arr (numpy array): EEG data of shape (20, 31, 60, 575)
                              (subjects, channels, stimuli, time points).
        metric (str): Distance metric to use for RDM computation (e.g., 'correlation', 'euclidean').

    Returns:
        eeg_rdms (numpy array): RDMs of shape (20, 60, 60, 575)
                                (subjects, stimuli x stimuli, time points).
    """
    n_subjects, n_channels, n_stimuli, n_timepoints = eeg_arr.shape
    eeg_rdms = np.zeros((n_subjects, n_stimuli, n_stimuli, n_timepoints))

    # Iterate over subjects
    for subj in range(n_subjects):
        print(f"Processing Subject {subj + 1}/{n_subjects}...")
        
        # Iterate over time points
        for t in range(n_timepoints):
            # Extract data at time point t for all stimuli (shape: 60, 31)
            data_at_t = eeg_arr[subj, :, :, t].T  # Transpose to shape (60, 31)
            
            # Compute pairwise distances (RDM) between stimuli
            rdm = pairwise_distances(data_at_t, metric=metric)  # Shape: (60, 60)
            
            # Store the RDM
            eeg_rdms[subj, :, :, t] = rdm

    return eeg_rdms


def extract_upper_triangle(matrix):
    """Extract the upper triangular part of a matrix as a 1D array, excluding the diagonal."""
    return matrix[np.triu_indices_from(matrix, k=1)]

def correlate_rdms(brain_rdms, eeg_rdms):
    """
    Correlate brain RDMs (motor or loc) with EEG RDMs over time for each subject.

    Parameters:
        brain_rdms (numpy array): Shape (20, 60, 60), brain RDMs (motor or loc).
        eeg_rdms (numpy array): Shape (20, 60, 60, 575), EEG RDMs.

    Returns:
        correlations (numpy array): Correlations of shape (20, 575).
    """
    n_subjects, _, _, n_timepoints = eeg_rdms.shape
    correlations = np.zeros((n_subjects, n_timepoints))

    # Loop over subjects
    for subj in range(n_subjects):
        brain_rdm_upper = extract_upper_triangle(brain_rdms[subj])  # Upper triangle of brain RDM

        # Loop over timepoints
        for t in range(n_timepoints):
            eeg_rdm_upper = extract_upper_triangle(eeg_rdms[subj, :, :, t])  # Upper triangle of EEG RDM

            # Compute Pearson correlation between upper triangles
            if not np.isnan(eeg_rdm_upper).any():  # Check for NaN values
                corr, _ = pearsonr(brain_rdm_upper, eeg_rdm_upper)
            else:
                corr = np.nan  # Assign NaN if invalid data
            correlations[subj, t] = corr

    return correlations

# Step 1. Generate subject level rsa

In [3]:
# Import mask
brain_mask = np.load("N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\mask.npy")
centers, neighbors = get_volume_searchlight(brain_mask, radius=5, threshold=0.5)

Finding searchlights...: 100%|██████████| 59159/59159 [00:07<00:00, 8054.20it/s]


Found 56949 searchlights


In [4]:
eeg_arr = np.load("../IAPS_EEG_RSA/eeg_raw.npy")
allsub_avg = np.load("../IAPS_Searchlight/allsub_avg.npy", allow_pickle=True)

In [7]:
eeg_arr.shape

(20, 31, 60, 575)

In [6]:
allsub_avg.shape

(20, 60, 153594)

In [5]:
eeg_rdms = compute_eeg_rdms(eeg_arr, metric='correlation')
print("Shape of EEG RDMs:", eeg_rdms.shape)


Processing Subject 1/20...
Processing Subject 2/20...
Processing Subject 3/20...
Processing Subject 4/20...
Processing Subject 5/20...
Processing Subject 6/20...
Processing Subject 7/20...
Processing Subject 8/20...
Processing Subject 9/20...
Processing Subject 10/20...
Processing Subject 11/20...
Processing Subject 12/20...
Processing Subject 13/20...
Processing Subject 14/20...
Processing Subject 15/20...
Processing Subject 16/20...
Processing Subject 17/20...
Processing Subject 18/20...
Processing Subject 19/20...
Processing Subject 20/20...
Shape of EEG RDMs: (20, 60, 60, 575)


In [8]:
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from joblib import Parallel, delayed

# Helper function
def upper_tri(matrix):
    return matrix[np.triu_indices_from(matrix, k=1)]

# Define condition splits
conditions = {
    'pleasant':   np.arange(0, 20),
    'neutral':    np.arange(20, 40),
    'unpleasant': np.arange(40, 60),
}

# Function to process a single subject for a given condition
def process_subject(subj, allsub_avg, eeg_rdms, centers, neighbors, brain_mask, n_timepoints, n_voxels, stim_indices, cond_name):
    """Process RSA for a single subject and one emotion condition"""
    
    # Subset fMRI data to this condition's stimuli
    subj_data = np.nan_to_num(allsub_avg[subj])  # (60, n_voxels)
    subj_data_cond = subj_data[stim_indices]       # (20, n_voxels)
    image_value = np.arange(subj_data_cond.shape[0])
    
    # Get searchlight RDMs using only the 20 condition stimuli
    SL_RDM = get_searchlight_RDMs(subj_data_cond, centers, neighbors, image_value, method='correlation')
    voxel_indices = SL_RDM.rdm_descriptors['voxel_index']
    
    # Subset EEG RDMs: original is (60, 60, T), take the condition sub-block (20, 20, T)
    eeg_rdm_full = eeg_rdms[subj]  # (60, 60, T)
    eeg_rdm_cond = eeg_rdm_full[np.ix_(stim_indices, stim_indices)]  # (20, 20, T)
    
    # Initialize output
    subj_searchlight_rsa = np.zeros((n_timepoints, n_voxels))
    
    for t in tqdm(range(n_timepoints), desc=f"Subj {subj} {cond_name}", leave=False):
        eeg_rdm_t = eeg_rdm_cond[:, :, t]
        eeg_vec = upper_tri(np.nan_to_num(eeg_rdm_t))
        
        eval_score = []
        for rdm in SL_RDM:
            sl_vec = np.nan_to_num(np.squeeze(rdm.dissimilarities))
            mask = ~np.isnan(eeg_vec) & ~np.isnan(sl_vec)
            if np.sum(mask) == 0:
                eval_score.append(np.nan)
            else:
                corr, _ = spearmanr(eeg_vec[mask], sl_vec[mask])
                eval_score.append(corr)
        
        subj_searchlight_rsa[t, :] = eval_score
    
    return subj_searchlight_rsa


# Main execution
n_subjects = allsub_avg.shape[0]
n_timepoints = eeg_rdms.shape[-1]
n_voxels = len(centers)

n_jobs = min(n_subjects, -1)

for cond_name, stim_indices in conditions.items():
    print(f"\n{'='*60}")
    print(f"Processing condition: {cond_name} (stimuli {stim_indices[0]}-{stim_indices[-1]})")
    print(f"{'='*60}")
    
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(process_subject)(
            subj, allsub_avg, eeg_rdms, centers, neighbors,
            brain_mask, n_timepoints, n_voxels, stim_indices, cond_name
        ) for subj in range(n_subjects)
    )
    
    searchlight_rsa = np.array(results)  # (n_subjects, n_timepoints, n_voxels)
    print(f"Searchlight RSA shape ({cond_name}):", searchlight_rsa.shape)
    
    np.save(f"searchlight_rsa_{cond_name}.npy", searchlight_rsa)
    print(f"Saved searchlight_rsa_{cond_name}.npy")

print("\nDone! Saved: searchlight_rsa_pleasant.npy, searchlight_rsa_neutral.npy, searchlight_rsa_unpleasant.npy")


Processing condition: pleasant (stimuli 0-19)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 28 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  20 | elapsed: 193.1min remaining: 772.5min
[Parallel(n_jobs=-1)]: Done   7 out of  20 | elapsed: 193.3min remaining: 359.0min
[Parallel(n_jobs=-1)]: Done  10 out of  20 | elapsed: 193.4min remaining: 193.4min
[Parallel(n_jobs=-1)]: Done  13 out of  20 | elapsed: 193.5min remaining: 104.2min
[Parallel(n_jobs=-1)]: Done  16 out of  20 | elapsed: 193.8min remaining: 48.4min
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed: 194.8min finished


Searchlight RSA shape (pleasant): (20, 575, 56949)
Saved searchlight_rsa_pleasant.npy

Processing condition: neutral (stimuli 20-39)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 28 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  20 | elapsed: 190.6min remaining: 762.3min
[Parallel(n_jobs=-1)]: Done   7 out of  20 | elapsed: 190.7min remaining: 354.1min
[Parallel(n_jobs=-1)]: Done  10 out of  20 | elapsed: 191.0min remaining: 191.0min
[Parallel(n_jobs=-1)]: Done  13 out of  20 | elapsed: 191.2min remaining: 103.0min
[Parallel(n_jobs=-1)]: Done  16 out of  20 | elapsed: 191.7min remaining: 47.9min
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed: 192.8min finished


Searchlight RSA shape (neutral): (20, 575, 56949)
Saved searchlight_rsa_neutral.npy

Processing condition: unpleasant (stimuli 40-59)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 28 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  20 | elapsed: 190.6min remaining: 762.3min
[Parallel(n_jobs=-1)]: Done   7 out of  20 | elapsed: 190.8min remaining: 354.3min
[Parallel(n_jobs=-1)]: Done  10 out of  20 | elapsed: 191.0min remaining: 191.0min
[Parallel(n_jobs=-1)]: Done  13 out of  20 | elapsed: 191.1min remaining: 102.9min
[Parallel(n_jobs=-1)]: Done  16 out of  20 | elapsed: 191.3min remaining: 47.8min
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed: 192.8min finished


Searchlight RSA shape (unpleasant): (20, 575, 56949)
Saved searchlight_rsa_unpleasant.npy

Done! Saved: searchlight_rsa_pleasant.npy, searchlight_rsa_neutral.npy, searchlight_rsa_unpleasant.npy


In [9]:
import numpy as np
import nibabel as nib
import pandas as pd
import os

# Define conditions
conditions = ['pleasant', 'neutral', 'unpleasant']

# Load voxel index map
voxel_center_id = pd.read_csv(
    r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\voxel_center_id.csv",
    header=0,
    index_col=0
)
voxel_indices = np.squeeze(np.array(voxel_center_id))

# Load template image for affine/header
tmp_image_path = 'N:/Experimental_Data/yujunchen/projects/IAPS_fMRI_RSA/fMRI_singletrial_betas/nifti/Nt1.img'
tmp_img = nib.load(tmp_image_path)

# Define 3D volume shape
x, y, z = 53, 63, 46

for cond in conditions:
    print(f"\n{'='*60}")
    print(f"Processing condition: {cond}")
    print(f"{'='*60}")

    # Create output folder
    out_dir = f"searchlight_rsa_{cond}"
    os.makedirs(out_dir, exist_ok=True)

    # Load condition-specific RSA results
    rsa_data = np.load(f"searchlight_rsa_{cond}.npy")  # (n_subjects, n_timepoints, n_voxels)
    n_subjects, n_timepoints, n_voxels = rsa_data.shape
    print(f"  Shape: {rsa_data.shape}")

    for subj_idx in range(n_subjects):
        print(f"  Subject {subj_idx+1}/{n_subjects}")

        rsa_4d = np.full((x * y * z, n_timepoints), np.nan)

        for t in range(n_timepoints):
            rsa_4d[voxel_indices, t] = rsa_data[subj_idx, t, :]

        rsa_4d_volume = rsa_4d.reshape((x, y, z, n_timepoints))

        rsa_img = nib.Nifti1Image(rsa_4d_volume, affine=tmp_img.affine, header=tmp_img.header)

        out_path = os.path.join(out_dir, f"rsa_{cond}_subject{subj_idx+1:02d}.nii.gz")
        nib.save(rsa_img, out_path)

    print(f"  Saved {n_subjects} files to {out_dir}/")

print("\nDone!")


Processing condition: pleasant
  Shape: (20, 575, 56949)
  Subject 1/20
  Subject 2/20
  Subject 3/20
  Subject 4/20
  Subject 5/20
  Subject 6/20
  Subject 7/20
  Subject 8/20
  Subject 9/20
  Subject 10/20
  Subject 11/20
  Subject 12/20
  Subject 13/20
  Subject 14/20
  Subject 15/20
  Subject 16/20
  Subject 17/20
  Subject 18/20
  Subject 19/20
  Subject 20/20
  Saved 20 files to searchlight_rsa_pleasant/

Processing condition: neutral
  Shape: (20, 575, 56949)
  Subject 1/20
  Subject 2/20
  Subject 3/20
  Subject 4/20
  Subject 5/20
  Subject 6/20
  Subject 7/20
  Subject 8/20
  Subject 9/20
  Subject 10/20
  Subject 11/20
  Subject 12/20
  Subject 13/20
  Subject 14/20
  Subject 15/20
  Subject 16/20
  Subject 17/20
  Subject 18/20
  Subject 19/20
  Subject 20/20
  Saved 20 files to searchlight_rsa_neutral/

Processing condition: unpleasant
  Shape: (20, 575, 56949)
  Subject 1/20
  Subject 2/20
  Subject 3/20
  Subject 4/20
  Subject 5/20
  Subject 6/20
  Subject 7/20
  Subje

# Step 2. Get the mean rsa across subjects

In [10]:
import numpy as np
import nibabel as nib
import os

conditions = ['pleasant', 'neutral', 'unpleasant']
n_subjects = 20

for cond in conditions:
    print(f"\nProcessing condition: {cond}")

    in_dir = f"searchlight_rsa_{cond}"
    file_list = [os.path.join(in_dir, f"rsa_{cond}_subject{i:02d}.nii.gz") for i in range(1, n_subjects + 1)]

    # Load first file for shape/affine
    img0 = nib.load(file_list[0])
    data_shape = img0.shape

    # Pre-allocate
    all_data = np.zeros((n_subjects, *data_shape), dtype=np.float32)

    for idx, fname in enumerate(file_list):
        img = nib.load(fname)
        all_data[idx] = img.get_fdata()

    # Nanmean over subjects
    mean_data = np.nanmean(all_data, axis=0)

    # Save mean image in the same subfolder
    mean_path = os.path.join(in_dir, f"rsa_{cond}_subject_mean.nii.gz")
    mean_img = nib.Nifti1Image(mean_data, affine=img0.affine, header=img0.header)
    nib.save(mean_img, mean_path)

    print(f"  Saved {mean_path}")

print("\nDone!")


Processing condition: pleasant


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\2643183503.py:26: RuntimeWarning: Mean of empty slice
  mean_data = np.nanmean(all_data, axis=0)


  Saved searchlight_rsa_pleasant\rsa_pleasant_subject_mean.nii.gz

Processing condition: neutral
  Saved searchlight_rsa_neutral\rsa_neutral_subject_mean.nii.gz

Processing condition: unpleasant
  Saved searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean.nii.gz

Done!


In [11]:
import numpy as np
import nibabel as nib
from nilearn.image import resample_img
import os

conditions = ['pleasant', 'neutral', 'unpleasant']

vx = np.array([2.0, 2.0, 2.0])

for cond in conditions:
    input_path = os.path.join(f"searchlight_rsa_{cond}", f"rsa_{cond}_subject_mean.nii.gz")
    output_path = os.path.join(f"searchlight_rsa_{cond}", f"rsa_{cond}_subject_mean_2mm.nii.gz")

    print(f"Resampling {input_path}...")

    img = nib.load(input_path)

    zooms = nib.affines.voxel_sizes(img.affine)[:3]
    fov = np.array(img.shape[:3]) * zooms
    target_shape = tuple(np.ceil(fov / vx).astype(int))

    A = img.affine[:3, :3]
    new_A = A @ np.diag(vx / zooms)
    target_affine = np.eye(4)
    target_affine[:3, :3] = new_A
    target_affine[:3, 3] = img.affine[:3, 3]

    hi = resample_img(
        img,
        target_affine=target_affine,
        target_shape=target_shape,
        interpolation="continuous",
    )

    hi32 = nib.Nifti1Image(hi.get_fdata(dtype=np.float32), hi.affine, hi.header)
    hi32.header.set_data_dtype(np.float32)
    nib.save(hi32, output_path)
    print(f"  Saved {output_path}")

print("\nDone!")

Resampling searchlight_rsa_pleasant\rsa_pleasant_subject_mean.nii.gz...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a b

  Saved searchlight_rsa_pleasant\rsa_pleasant_subject_mean_2mm.nii.gz
Resampling searchlight_rsa_neutral\rsa_neutral_subject_mean.nii.gz...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a b

  Saved searchlight_rsa_neutral\rsa_neutral_subject_mean_2mm.nii.gz
Resampling searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean.nii.gz...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1658382069.py:28: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a b

  Saved searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean_2mm.nii.gz

Done!


In [12]:
import os
import numpy as np
import nibabel as nib
from nilearn.image import resample_img

conditions = ['pleasant', 'neutral', 'unpleasant']
n_subjects = 20
target_resolution = 1.0  # mm

for cond in conditions:
    print(f"\nProcessing condition: {cond}")

    in_dir = f"searchlight_rsa_{cond}"
    file_list = [
        os.path.join(in_dir, f"rsa_{cond}_subject{i:02d}.nii.gz")
        for i in range(1, n_subjects + 1)
    ]

    # -----------------------------
    # Check files exist
    # -----------------------------
    missing_files = [f for f in file_list if not os.path.exists(f)]
    if missing_files:
        print("Missing files:")
        for f in missing_files:
            print(f"  {f}")
        continue

    # -----------------------------
    # Load first file for reference
    # -----------------------------
    img0 = nib.load(file_list[0])
    data_shape = img0.shape
    print(f"Original shape: {data_shape}")

    current_voxel_sizes = nib.affines.voxel_sizes(img0.affine)[:3]
    print(f"Current voxel sizes: {current_voxel_sizes} mm")

    # -----------------------------
    # Load all subjects
    # -----------------------------
    all_data = np.zeros((n_subjects, *data_shape), dtype=np.float32)

    for idx, fname in enumerate(file_list):
        img = nib.load(fname)
        all_data[idx] = img.get_fdata()

    # -----------------------------
    # Average across subjects
    # -----------------------------
    mean_data = np.nanmean(all_data, axis=0)

    mean_path = os.path.join(in_dir, f"rsa_{cond}_subject_mean.nii.gz")
    mean_img = nib.Nifti1Image(mean_data, affine=img0.affine, header=img0.header)
    nib.save(mean_img, mean_path)
    print(f"Saved mean image: {mean_path}")

    # -----------------------------
    # Resample mean image to 1 mm
    # -----------------------------
    scale_factors = current_voxel_sizes / target_resolution

    target_affine = img0.affine.copy()
    target_affine[:3, :3] = img0.affine[:3, :3] / scale_factors[:, np.newaxis]

    current_shape = img0.shape[:3]
    target_shape = tuple(
        int(np.round(s * sf)) for s, sf in zip(current_shape, scale_factors)
    )

    print(f"Target voxel size: [1.0, 1.0, 1.0] mm")
    print(f"Target shape: {target_shape}")
    print("Resampling mean image to 1 mm...")

    resampled = resample_img(
        mean_img,
        target_affine=target_affine,
        target_shape=target_shape,
        interpolation='continuous'
    )

    resampled_path = os.path.join(in_dir, f"rsa_{cond}_subject_mean_1mm.nii.gz")
    nib.save(resampled, resampled_path)
    print(f"Saved resampled image: {resampled_path}")

    # -----------------------------
    # Report file sizes
    # -----------------------------
    original_size = os.path.getsize(mean_path) / (1024 * 1024)
    new_size = os.path.getsize(resampled_path) / (1024 * 1024)

    print(f"Mean image size: {original_size:.1f} MB")
    print(f"1 mm image size: {new_size:.1f} MB")

print("\nDone!")


Processing condition: pleasant
Original shape: (53, 63, 46, 575)
Current voxel sizes: [3. 3. 3.] mm


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:51: RuntimeWarning: Mean of empty slice
  mean_data = np.nanmean(all_data, axis=0)


Saved mean image: searchlight_rsa_pleasant\rsa_pleasant_subject_mean.nii.gz
Target voxel size: [1.0, 1.0, 1.0] mm
Target shape: (159, 189, 138)
Resampling mean image to 1 mm...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: RuntimeWarning: NaNs or infinite values are present in the data passed to resa

Saved resampled image: searchlight_rsa_pleasant\rsa_pleasant_subject_mean_1mm.nii.gz
Mean image size: 120.8 MB
1 mm image size: 3193.0 MB

Processing condition: neutral
Original shape: (53, 63, 46, 575)
Current voxel sizes: [3. 3. 3.] mm


...

Saved resampled image: searchlight_rsa_neutral\rsa_neutral_subject_mean_1mm.nii.gz
Mean image size: 121.7 MB
1 mm image size: 3216.2 MB

Processing condition: unpleasant
Original shape: (53, 63, 46, 575)
Current voxel sizes: [3. 3. 3.] mm


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:51: RuntimeWarning: Mean of empty slice
  mean_data = np.nanmean(all_data, axis=0)


Saved mean image: searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean.nii.gz
Target voxel size: [1.0, 1.0, 1.0] mm
Target shape: (159, 189, 138)
Resampling mean image to 1 mm...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  resampled = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\138917371.py:75: RuntimeWarning: NaNs or infinite values are present in the data passed to resa

Saved resampled image: searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean_1mm.nii.gz
Mean image size: 120.4 MB
1 mm image size: 3182.6 MB

Done!


In [13]:
import os
import numpy as np
import nibabel as nib
from nibabel import gifti
from skimage import measure


def create_fsleyes_compatible_4d_surface(input_4d_nifti, output_prefix):
    """
    Create FSLeyes-compatible surface + timeseries GIFTI files
    from a 4D NIfTI file.
    """
    print(f"\nLoading 4D data: {input_4d_nifti}")
    img = nib.load(input_4d_nifti)
    data = img.get_fdata()
    print(f"Shape: {data.shape}")

    if data.ndim != 4:
        raise ValueError(f"Expected 4D NIfTI, but got shape {data.shape}")

    # Create surface from temporal mean
    mean_vol = np.mean(np.abs(data), axis=3)

    nonzero_vals = mean_vol[mean_vol > 0]
    if nonzero_vals.size == 0:
        raise ValueError(f"Mean volume is all zeros in {input_4d_nifti}")

    threshold = np.percentile(nonzero_vals, 20)
    print(f"Surface threshold: {threshold:.6f}")

    # Extract surface mesh
    print("Creating surface mesh...")
    verts, faces, _, _ = measure.marching_cubes(
        mean_vol,
        level=threshold,
        spacing=nib.affines.voxel_sizes(img.affine)[:3]
    )

    # Transform to world coordinates
    verts_hom = np.column_stack([verts, np.ones(len(verts))])
    verts_world = verts_hom.dot(img.affine.T)[:, :3]

    # -----------------------------
    # SURFACE FILE (geometry only)
    # -----------------------------
    print("Creating surface geometry file...")

    coords = gifti.GiftiDataArray(
        data=verts_world.astype(np.float32),
        intent='NIFTI_INTENT_POINTSET',
        datatype='NIFTI_TYPE_FLOAT32'
    )

    triangles = gifti.GiftiDataArray(
        data=faces.astype(np.int32),
        intent='NIFTI_INTENT_TRIANGLE',
        datatype='NIFTI_TYPE_INT32'
    )

    surface_gii = gifti.GiftiImage()
    surface_gii.add_gifti_data_array(coords)
    surface_gii.add_gifti_data_array(triangles)

    surface_file = f"{output_prefix}_surface.surf.gii"
    nib.save(surface_gii, surface_file)
    print(f"Saved surface: {surface_file}")

    # -----------------------------
    # TIMESERIES FILE
    # -----------------------------
    print("Creating 4D overlay file...")
    n_timepoints = data.shape[3]
    vertex_data = np.zeros((len(verts), n_timepoints), dtype=np.float32)

    # Sample time series at each vertex
    for i, vert in enumerate(verts):
        vox = np.round(vert).astype(int)
        vox = np.clip(vox, 0, np.array(data.shape[:3]) - 1)
        vertex_data[i, :] = data[vox[0], vox[1], vox[2], :]

    timeseries_gii = gifti.GiftiImage()
    timeseries_gii.meta = gifti.GiftiMetaData()
    timeseries_gii.meta.data.append(
        gifti.GiftiNVPairs(name='TimeStep', value='1.0')
    )

    for t in range(n_timepoints):
        tdata = gifti.GiftiDataArray(
            data=vertex_data[:, t].astype(np.float32),
            intent='NIFTI_INTENT_NONE',
            datatype='NIFTI_TYPE_FLOAT32'
        )
        timeseries_gii.add_gifti_data_array(tdata)

    timeseries_file = f"{output_prefix}_timeseries.func.gii"
    nib.save(timeseries_gii, timeseries_file)
    print(f"Saved timeseries: {timeseries_file}")

    print("\n" + "=" * 60)
    print("FILES CREATED SUCCESSFULLY")
    print("=" * 60)
    print("To load in FSLeyes:")
    print(f"1. File → Add from file → {surface_file}")
    print(f"2. Then: File → Add from file → {timeseries_file}")
    print("The surface will load first, then the time series will be applied to it.")

    return surface_file, timeseries_file


# =========================================================
# Batch over conditions
# =========================================================
conditions = ['pleasant', 'neutral', 'unpleasant']

for cond in conditions:
    in_dir = f"searchlight_rsa_{cond}"
    input_nii = os.path.join(in_dir, f"rsa_{cond}_subject_mean_1mm.nii.gz")
    output_prefix = os.path.join(in_dir, f"rsa_{cond}_fsleyes")

    print("\n" + "#" * 70)
    print(f"Processing condition: {cond}")
    print("#" * 70)

    if not os.path.exists(input_nii):
        print(f"Input file not found: {input_nii}")
        continue

    try:
        create_fsleyes_compatible_4d_surface(input_nii, output_prefix)
    except Exception as e:
        print(f"Error processing {cond}: {e}")

print("\nDone!")


######################################################################
Processing condition: pleasant
######################################################################

Loading 4D data: searchlight_rsa_pleasant\rsa_pleasant_subject_mean_1mm.nii.gz
Shape: (159, 189, 138, 575)
Surface threshold: 0.015883
Creating surface mesh...
Creating surface geometry file...
Saved surface: searchlight_rsa_pleasant\rsa_pleasant_fsleyes_surface.surf.gii
Creating 4D overlay file...


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1682784106.py:77: RuntimeWarning: invalid value encountered in cast
  vox = np.round(vert).astype(int)
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1682784106.py:83: DeprecationWarning: The data attribute is deprecated. Use GiftiMetaData object directly as a dict.

* deprecated from version: 4.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 6.0
  timeseries_gii.meta.data.append(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_31980\1682784106.py:84: DeprecationWarning: GiftiNVPairs objects are deprecated. Use the GiftiMetaData object as a dict, instead.

* deprecated from version: 4.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 6.0
  gifti.GiftiNVPairs(name='TimeStep', value='1.0')


Saved timeseries: searchlight_rsa_pleasant\rsa_pleasant_fsleyes_timeseries.func.gii

FILES CREATED SUCCESSFULLY
To load in FSLeyes:
1. File → Add from file → searchlight_rsa_pleasant\rsa_pleasant_fsleyes_surface.surf.gii
2. Then: File → Add from file → searchlight_rsa_pleasant\rsa_pleasant_fsleyes_timeseries.func.gii
The surface will load first, then the time series will be applied to it.

######################################################################
Processing condition: neutral
######################################################################

Loading 4D data: searchlight_rsa_neutral\rsa_neutral_subject_mean_1mm.nii.gz
Shape: (159, 189, 138, 575)
Surface threshold: 0.014111
Creating surface mesh...
Creating surface geometry file...
Saved surface: searchlight_rsa_neutral\rsa_neutral_fsleyes_surface.surf.gii
Creating 4D overlay file...
Saved timeseries: searchlight_rsa_neutral\rsa_neutral_fsleyes_timeseries.func.gii

FILES CREATED SUCCESSFULLY
To load in FSLeyes:
1. File 

Remove NaN

In [16]:
import os
import nibabel as nib
import numpy as np

base_dir = "N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion"
emotions = ["neutral", "pleasant", "unpleasant"]

for emo in emotions:
    in_file = os.path.join(
        base_dir,
        f"searchlight_rsa_{emo}",
        f"rsa_{emo}_subject_mean.nii.gz"
    )
    out_file = os.path.join(
        base_dir,
        f"searchlight_rsa_{emo}",
        f"rsa_{emo}_subject_mean_nonan.nii.gz"
    )

    if not os.path.exists(in_file):
        print(f"Missing file, skipping: {in_file}")
        continue

    print(f"Processing {emo}...")

    img = nib.load(in_file)
    data = img.get_fdata()
    n_nans = np.isnan(data).sum()
    print(f"NaNs found: {n_nans}")

    data = np.nan_to_num(data, nan=0.0)

    out = nib.Nifti1Image(data.astype(np.float32), img.affine, img.header)
    nib.save(out, out_file)

    print(f"Saved: {out_file}\n")

print("All emotions finished.")

Processing neutral...
NaNs found: 55570875
Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\searchlight_rsa_neutral\rsa_neutral_subject_mean_nonan.nii.gz

Processing pleasant...
NaNs found: 55570875
Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\searchlight_rsa_pleasant\rsa_pleasant_subject_mean_nonan.nii.gz

Processing unpleasant...
NaNs found: 55570875
Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\searchlight_rsa_unpleasant\rsa_unpleasant_subject_mean_nonan.nii.gz

All emotions finished.


# Step 3. Create surface mesh files

run create_surface.sh


# Step 4. Combine per frame gii into muti frame gii 

In [17]:
from pathlib import Path
import nibabel as nib
import numpy as np

base = Path(r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion")
emotions = ["neutral", "pleasant", "unpleasant"]

def merge(work, hemi, outname):
    files = sorted(work.glob(f"{hemi}.tp*.func.gii"))
    print(f"{work.name} | {hemi}: {len(files)} files -> {outname}")

    if len(files) == 0:
        print(f"  No files found for {work} / {hemi}, skipping.")
        return

    first = nib.load(str(files[0]))
    darrays = []

    for f in files:
        img = nib.load(str(f))
        data = img.darrays[0].data.astype(np.float32)
        darrays.append(nib.gifti.GiftiDataArray(data))

    out = nib.gifti.GiftiImage(
        darrays=darrays,
        meta=first.meta,
        labeltable=first.labeltable
    )

    nib.save(out, str(work / outname))
    print(f"  Saved: {work / outname}")

for emo in emotions:
    work = base / f"fs_rsa_{emo}"
    if not work.exists():
        print(f"Missing folder: {work}")
        continue

    merge(work, "lh", f"lh.rsa_{emo}.func.gii")
    merge(work, "rh", f"rh.rsa_{emo}.func.gii")

print("All emotions finished.")

fs_rsa_neutral | lh: 574 files -> lh.rsa_neutral.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\lh.rsa_neutral.func.gii
fs_rsa_neutral | rh: 574 files -> rh.rsa_neutral.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\rh.rsa_neutral.func.gii
fs_rsa_pleasant | lh: 574 files -> lh.rsa_pleasant.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\lh.rsa_pleasant.func.gii
fs_rsa_pleasant | rh: 574 files -> rh.rsa_pleasant.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\rh.rsa_pleasant.func.gii
fs_rsa_unpleasant | lh: 574 files -> lh.rsa_unpleasant.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\lh.rsa_unpleasant.func.gii
fs_rsa_unpleasant | rh: 574 files -> rh.rsa_unpleasant.func.gii
  Saved: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpl

# Step 5. Inflated surface mesh, multiframe (WSL)

In [ ]:
# One-off: convert inflated meshes to GIFTI (geometry)
mris_convert $FREESURFER_HOME/subjects/fsaverage/surf/lh.inflated lh.inflated.surf.gii
mris_convert $FREESURFER_HOME/subjects/fsaverage/surf/rh.inflated rh.inflated.surf.gii


# Step 6. Inflated functional surface 

run create_inflated_surface.sh

# Step 7. Combine Inflated surface functional into multiframe 

In [21]:
from pathlib import Path
import re
import nibabel as nib
from nibabel.gifti import GiftiImage, GiftiDataArray
from nibabel.nifti1 import intent_codes

base = Path(r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion")
emotions = ["neutral", "pleasant", "unpleasant"]

# Natural sort by numeric frame index: tp000, tp001, ...
def natural_key(p: Path):
    m = re.search(r'\.tp(\d+)\.', p.name)
    return int(m.group(1)) if m else p.name

def combine_scalar_frames(work: Path, pattern: str, outname: str):
    files = sorted(work.glob(pattern), key=natural_key)
    if not files:
        print(f"No files match {work / pattern}, skipping.")
        return

    scalars = []
    nverts = None

    for f in files:
        img = nib.load(str(f))

        # keep only non-geometry arrays
        darrays = [
            da for da in img.darrays
            if da.intent not in (
                intent_codes['NIFTI_INTENT_POINTSET'],
                intent_codes['NIFTI_INTENT_TRIANGLE']
            )
        ]

        if len(darrays) != 1:
            raise ValueError(f"{f.name}: expected 1 scalar array, found {len(darrays)}")

        arr = darrays[0].data
        if nverts is None:
            nverts = arr.shape[0]
        elif arr.shape[0] != nverts:
            raise ValueError(f"Vertex count mismatch in {f.name}: {arr.shape[0]} vs {nverts}")

        scalars.append(
            GiftiDataArray(
                arr.astype("float32"),
                intent=intent_codes['NIFTI_INTENT_SHAPE']
            )
        )

    out = GiftiImage(darrays=scalars)
    out_path = work / outname
    nib.save(out, str(out_path))
    print(f"Wrote {out_path} with {len(scalars)} frames (nverts={nverts})")

for emo in emotions:
    work = base / f"fs_rsa_{emo}"
    if not work.exists():
        print(f"Missing folder: {work}, skipping.")
        continue

    print(f"\nProcessing emotion: {emo}")
    combine_scalar_frames(
        work,
        "lh.tp*.inflated.func.gii",
        f"lh.rsa_{emo}.inflated.multiframe.func.gii"
    )
    combine_scalar_frames(
        work,
        "rh.tp*.inflated.func.gii",
        f"rh.rsa_{emo}.inflated.multiframe.func.gii"
    )

print("\nAll emotions finished.")


Processing emotion: neutral
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\lh.rsa_neutral.inflated.multiframe.func.gii with 574 frames (nverts=163842)
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\rh.rsa_neutral.inflated.multiframe.func.gii with 574 frames (nverts=163842)

Processing emotion: pleasant
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\lh.rsa_pleasant.inflated.multiframe.func.gii with 574 frames (nverts=163842)
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\rh.rsa_pleasant.inflated.multiframe.func.gii with 574 frames (nverts=163842)

Processing emotion: unpleasant
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\lh.rsa_unpleasant.inflated.multiframe.func.gii with 574 frames (nverts=163842)
Wrote N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\rh.rsa_unpleasant.inflat

In [22]:
import nibabel as nib
import numpy as np

files = [
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\lh.rsa_neutral.inflated.multiframe.func.gii",
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\rh.rsa_neutral.inflated.multiframe.func.gii",
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\lh.rsa_pleasant.inflated.multiframe.func.gii",
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\rh.rsa_pleasant.inflated.multiframe.func.gii",
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\lh.rsa_unpleasant.inflated.multiframe.func.gii",
r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\rh.rsa_unpleasant.inflated.multiframe.func.gii",
]

for f in files:
    gii = nib.load(f)
    data = np.array([d.data for d in gii.darrays])

    print("\n", f)
    print("shape:", data.shape)
    print("min :", np.nanmin(data))
    print("max :", np.nanmax(data))
    print("mean:", np.nanmean(data))


 N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\lh.rsa_neutral.inflated.multiframe.func.gii
shape: (574, 163842)
min : -0.11054418
max : 0.10860564
mean: -7.244376e-05

 N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_neutral\rh.rsa_neutral.inflated.multiframe.func.gii
shape: (574, 163842)
min : -0.106410585
max : 0.12175216
mean: 0.001401055

 N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\lh.rsa_pleasant.inflated.multiframe.func.gii
shape: (574, 163842)
min : -0.10043629
max : 0.16283202
mean: 0.015720144

 N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_pleasant\rh.rsa_pleasant.inflated.multiframe.func.gii
shape: (574, 163842)
min : -0.12567495
max : 0.15877683
mean: 0.018797208

 N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\fs_rsa_unpleasant\lh.rsa_unpleasant.inflated.multiframe.func.gii
shape: (574, 163842)
min : -0.07885857
max : 0.13419792
mean: 0.019033557

 N:\Ex